In [2]:
# DQN으로 Cart Pole 학습
# 핵심 구성요소 : Q-Network,
# Target Network : Q값 계산용 네트워크를 하나 더 두고, 일정 주기마다 Q-Network의 가중치를 복사해서 사용한다
# Experience Replay - 매 번의 경험을 (s, a, r, s', done) 형태로 저장 후 무작위로 샘플링하여 학습 진행

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
from collections import deque
import random

In [ ]:
# 강화학습
# agent는 state를 받아 action을 선택하고, 그에 따른 reward를 받으며 학습한다
# 학습 목표 : agent가 총 보상을 최대화하는 방향으로 policy를 학습한다.
env = gym.make('CartPole-v1') # 카트에 막대를 수직으로 세운 체 좌우로 움직여 균형을 유지하는 환경 제공
num_actions = env.action_space.n # 환경에서 가능한 행동의 개수 (카트를 왼쪽, 오른쪽으로 미는 두 가지 행동)
state_dim = env.observation_space.shape[0] # 환경 상태 공간의 차원 (카트 위치, 속도, 막대 각도, 각속도)
# print(num_actions) # 2 : 왼쪽, 오른쪽
# print(state_dim) # 4 : 카트위치, 카트속도, 막대각도, 막대각속도

# DQN 모델 정의
def create_model():
  model = Sequential([
      Input(shape=(state_dim,)),
      Dense(64, activation='relu'),
      Dense(64, activation='relu'),
      Dense(num_actions, activation='linear')    # 출력층 : Q값(정량적) 출력, 각 행동에 대한 Q값을 나타내는 뉴런
  ])

  model.compile(optimizer=Adam(learning_rate=1e-4), loss='mse')
  # 목표값(target) : reward + $\gamma$ * maxQ(s', a')
  # 예측값(prediction) : Q(s, a)
  # 예측값과 목표값 차이를 최소화
  return model

model = create_model()    # 주 네트워크(main) 학습을 직접 수행하는 신경망
target_model = create_model()     # 타겟 네트워크. 학습 중인 모델과는 별개로 유지되는 Q-Network
# Q-Learning의 안정성 향상을 위해 사용하는 '고정된 Q값 계산용 네트워크'
target_model.set_weights(model.get_weights()) # 초기에는 타겟 네트워크의 가중치를 주 네트워크와 동일하게 설정

# 하이퍼 파라미터
gamma = 0.99 # 감가율 (discount factor): 미래 보상을 얼마나 중요하게 고려할지 결정
epsilon = 1.0 # 탐험 시작 확률 (epsilon-greedy exploration)
epsilon_min = 0.05 # 탐험 최소 확률
epsilon_decay = 0.995 # 탐험 확률 감소율
batch_size = 64    # 경험 리플레이에서 랜덤하게 꺼내서 학습에 사용하는 샘플 수

# 경험 리플레이 버퍼 : 양방향 큐(FIFO) 자료구조로 append(), popleft() 사용
memory = deque(maxlen=5000)    # 경험 재사용 ; 5000개 넘으면 처음
episodes = 50    # 200 ~ 500

# Target Q-Network 갱신 주기
update_target_every = 5    # 5 에피소드마다 갱신
reward_list = []

# 학습 루핑
for ep in range(episodes):
  state, _ = env.reset()
  total_reward = 0    # 한 에피소드에서 받는 보상의 총합
  done = False    # 종료 조건

  while not done:    # 막대가 넘어진 경우 시간 초과인 경우 반복 종료
    state_input = np.reshape(state, [1, state_dim])    # 1차원을 2차원으로 reshape. DQN에서 state를 신경망 입력 형식에 맞게 변형

    if np.random.rand() < epsilon:
      action = np.random.choice(num_actions)    # 랜덤 행동 선택
    else:
      q_values = model.predict(state_input, verbose=0)
      action = np.argmax(q_values[0])    # Q값이 가장 큰 행동 선택

    next_state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    modified_reward = reward if not done else -10
    memory.append((state, action, modified_reward, next_state, done))
    state = next_state
    total_reward += reward

    # 학습 : 일정 수 이상 경험이 쌓이면 학습을 시작. 이때 벨만 방정식 기반의 Q값 갱신
    if len(memory) > batch_size:
      minibatch = random.sample(memory, batch_size)
      states, targets = [], []    # 상태입력값, Q값 target을 저장할 배열

      for s,a,r,s_next, d in minibatch:
        s_input = np.reshape(s, [1, state_dim])
        s_next_input = np.reshape(s_next, [1, state_dim])
        target = model.predict(s_input, verbose=0)[0]    # 차원 축소 : target은 [Q(s,0), Q(s,1)] 형태
        # print(target)

        if d:
          target[a] = r    # 종료 상태면 미래 보상이 없다.
        else:
          t_next = target_model.predict(s_next_input, verbose=0)[0]
          # Q(s,a)는 즉시 보상 + 미래 최대 Q값
          target[a] = r + gamma * np.max(t_next)    # 벨만 방정식

        states.append(s)    # 입력 데이터 리스트에 현재 상태 저장
        targets.append(target)    # 정답 Q값 저장용 벡터에 저장

      # print("states :", np.array(states))
      # print("targets :", np.array(targets))

      model.fit(np.array(states), np.array(targets), epochs=1, verbose=1)
      # 여기까지 : 리플레이 버퍼에서 무작위로 batch 꺼냄 -> sample에 대해서 Q(s, a) 갱신 ->
          # 전체 states, target을 모아 model.fit()

  # if, while 탈출
  reward_list.append(total_reward)    # 한 에피소드 동안 받은 보상 누적
  if epsilon > epsilon_min:
    epsilon *= epsilon_decay
    epsilon = max(epsilon, epsilon_min)    # 더 이상 줄어들지 않을 최소 탐험 비율 지정

  # 타겟 모델 갱신 target network
  if ep % update_target_every == 0:    # 5번 마다 갱신
    target_model.set_weights(model.get_weights())

  if ep % 10 == 0:
    print(f"Episode: {ep}, Total Reward: {total_reward:.1f}, Epsilon: {epsilon:.3f}")

# end for


Episode: 0, Total Reward: 39.0, Epsilon: 0.995
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - loss: 2.0997
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 2.1152
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 2.0393
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 3.0820
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 2.0914
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 1.4973
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 3.0021
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 2.0224
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 3.5134
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 1.5083
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 1.9358
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 1.9505
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 2.5414
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 2.5275
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 2.9053
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 2.3542
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 2.2949
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 3.4489

In [ ]:
# 보상 시각화
plt.figure(figsize=(10, 4))

# 이동평균 계산(노이즈 제거용)
def moving_average(data, window_size=10):
  return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

data = np.array([1,2,3,4,5])
window_size = 3
avg = np.convolve(data, np.ones(window_size)/window_size, mode='valid')
print(avg)

plt.plot(reward_list, label='Reward per Episode')
plt.plot(moving_average(reward_list), label='Moving avg', color='red')
plt.title('DQN Cartpole reward')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 모델 저장
model.save('dqn_model.keras')
print("모델 저장 성공!")

In [ ]:
# 애니메이션
from matplotlib.pathes import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from tensorflow import keras

env = gym.make('CartPole-v1', render_mode=None)
model = keras.models.load_model('dqn_model.keras')
state_dim = env.observation_space.shape[0]
num_actions = env.action_space.n

flat_states = []
episode_labels = []

state, _ = env.reset()
done = False
ep_num = 0

while not done:
  flat_states.append(state.copy())
  episode_labels.append(ep_num)
  state_input = np.reshape(state, [1, state_dim])
  q_values = model.predict(state_input, verbose=0)

  action = np.argmax(q_values[0])
  next_state, reward, terminated, truncated, _ = env.step(action)
  state = next_state
  done = terminated or truncated
env.close()

frame_count = len(flat_states)    # 애니메이션에 사용될 총 프레임 수
# print(frame_count)    # 9798장에 걸쳐서 에이전트가 환경에서 행동했다.

fig, ax = plt.subplots()
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-0.5, 1.5)    # 카트 위치 범위
ax.set_title("Cart Simulation")
ax.set_xlabel("Cart Position")
ax.set_ylabel("Stick Height")

# Cart
cart_width = 0.4
cart_height = 0.2
cart_y = 0.0
cart_rect = Rectangle((0,0), cart_width, cart_height, color='black')
ax.add_patch(cart_rect)

# Pole
pole_len = 1.0  # Define pole length
line_list = ax.plot([], [], 'r-', linewidth=4)
pole_line = line_list[0]

episode_text = ax.text(0.05, 1.4, '', transform=ax.transData, color='blue')

def update(frame):
  x = flat_states[frame][0]
  theta = flat_states[frame][2]    # 현재 프레임에서 막대 각도(rad 단위)
  ep_num = episode_labels[frame]
  cart_rect.set_xy((x - cart_width / 2, cart_y))

  # pole 끝 좌표 계산
  x_start = x
  y_start = cart_y + cart_height
  x_end = x_start + pole_len * np.sin(theta)
  y_end = y_start + pole_len * np.cos(theta)
  pole_line.set_data([x_start, x_end], [y_start, y_end])

  episode_text.set_text(f"Episode : {ep_num}")
  return cart_rect, pole_line, episode_text

ani = FuncAnimation(fig, update, frames=frame_count, interval=50, repeat=False)
plt.close(fig)
display(HTML(ani.to_jshtml()))
